# Creator Revenue Prediction — Pure PySpark
Notebook tái lập pipeline Spark DataFrame, Spark ML training, evaluation và artifact export. Không sử dụng pandas hoặc scikit-learn.

In [ ]:
!java -version
!python --version
!pip install -q pyspark==4.2.0 streamlit==1.63.0

In [ ]:
import os, shutil
repo_dir = '/content/CreatorRevenuePrediction-PySpark'
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)
!git clone https://github.com/kieu-collab/CreatorRevenuePrediction-PySpark.git {repo_dir}
%cd {repo_dir}

In [ ]:
!python train.py --data data/creator_campaign.csv --artifact-dir artifacts --master 'local[*]' --driver-memory 4g --hash-features 128

In [ ]:
import csv, json
with open('artifacts/metadata.json', encoding='utf-8') as handle:
    metadata = json.load(handle)
with open('artifacts/metrics.csv', encoding='utf-8') as handle:
    metrics = list(csv.DictReader(handle))
print(json.dumps({k: metadata[k] for k in ['engine', 'spark_version', 'rows', 'unique_creators', 'best_model', 'successful_model_count']}, ensure_ascii=False, indent=2))
for index, row in enumerate(metrics, 1):
    print(index, row['Model'], 'MAE=', f"{float(row['MAE']):,.0f}", 'RMSE=', f"{float(row['RMSE']):,.0f}", 'MAPE=', f"{float(row['MAPE']):.2f}%", 'R2=', f"{float(row['R2']):.4f}")

In [ ]:
from src.prediction import load_pipeline_model, predict_one
from src.spark_session import create_spark_session
spark = create_spark_session('local[2]', 'CreatorRevenueSmokeTest')
model_path = 'artifacts/' + metadata['available_models'][metadata['best_model']]
model = load_pipeline_model(model_path)
payload = {
    'creator_id': 'demo_creator', 'creator_name': 'Beauty Creator',
    'brand_name': 'Demo Brand', 'creator_niche': 'Skincare',
    'product_category': 'Skincare', 'campaign_date': '2026-09-15',
    'followers': 180000.0, 'avg_views': 65000.0, 'engagement_rate': 0.055,
    'product_price': 329000.0, 'discount_rate': 0.10,
    'campaign_duration_days': 14.0, 'planned_posts': 4.0,
    'planned_live_sessions': 1.0, 'products_promoted': 2.0,
    'historical_conversion_rate': 0.025, 'brand_fit_score': 85.0,
    'creator_cost': 15000000.0, 'gross_margin_rate': 0.40
}
print(predict_one(payload, spark, model))
spark.stop()

In [ ]:
!zip -qr /content/CreatorRevenuePySpark_artifacts.zip artifacts
from google.colab import files
files.download('/content/CreatorRevenuePySpark_artifacts.zip')